Label Construction & Dataset Preparation

In [3]:
import numpy as np
import pandas as pd
import os
from rasterio.windows import    Window
from pyproj import Transformer
from tqdm import tqdm

In [6]:
with open("Output/filtered_images.txt", "r") as f:
    image_files = f.read().splitlines()


In [7]:
print("lenth of filtered images", len(image_files))

lenth of filtered images 8015


#MAP ESA


In [ ]:
import rasterio
landcover = rasterio.open("Raw_Data/worldcover_bbox_delhi_ncr_2021.tif")

esa_mapping = {10:"Vegetation",
                20:"Shrubland", 
                30:"Grassland",
                  40:"Cropland",
                    50:"Built-up",
                      80:"Water"}

transformer = Transformer.from_crs("EPSG:4326", landcover.crs, always_xy=True)

LABEL MAPPING

In [ ]:
valid_images = []
labels = []

for path in tqdm(image_files):
    filename = os.path.basename(path)
    parts = filename.replace(".png","").split("_")
    try:
        lat = float(parts[-2])
        lon = float(parts[-1])
    except:
        continue

    x, y = transformer.transform(lon, lat)
    row, col = landcover.index(x, y)

    if row < 64 or col < 64:
        continue

    window = Window(col-64, row-64, 128, 128)  # for 128*128
    patch = landcover.read(1, window=window)

    if patch.shape != (128,128):
        continue

    values, counts = np.unique(patch, return_counts=True)
    dominant_class = values[np.argmax(counts)]
    if dominant_class == 0: 
        continue

    label = esa_mapping.get(dominant_class, "Others")
    valid_images.append(path)
    labels.append(label)

100%|██████████| 8015/8015 [00:03<00:00, 2324.39it/s]


now make a dataframe

In [11]:
import pandas as pd
df = pd.DataFrame({"image":valid_images , "label": labels})


In [17]:
print("dataset lenth",len(df))
df.head(10)


dataset lenth 8015


,image,label
0,Raw_Data/rgb\28.2056_76.8558.png,Built-up
1,Raw_Data/rgb\28.2056_76.8646.png,Built-up
2,Raw_Data/rgb\28.2056_76.8734.png,Built-up
3,Raw_Data/rgb\28.2056_76.8822.png,Built-up
4,Raw_Data/rgb\28.2056_76.8910.png,Cropland
5,Raw_Data/rgb\28.2056_76.8943.png,Cropland
6,Raw_Data/rgb\28.2056_76.9057.png,Cropland
7,Raw_Data/rgb\28.2056_76.9145.png,Cropland
8,Raw_Data/rgb\28.2056_76.9233.png,Cropland
9,Raw_Data/rgb\28.2056_76.9321.png,Cropland


Make a Train test split


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
train_df , test_df = train_test_split(df , test_size=0.4 , stratify=df["label"], random_state=40)

In [24]:
train_df
len(train_df)


4809

In [25]:
len(test_df)

3206

now we will save dataset to we can make model on this dataset


In [27]:
df.to_csv("Output/dataset.csv", index=False)
train_df.to_csv("Output/train_dataset.csv", index=False)
test_df.to_csv("Output/text_dataset.csv", index=False)

